# 实验报告 03:正则表达式与网络结构化数据

**学号**:20244080134　**姓名**:平吉佳

In [1]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080134

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境 ] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[ 学号 ]       {STUDENT_ID}")
print(f"[ 机器指纹 ] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境 ] exp_env
[ 学号 ]       20244080134
[ 机器指纹 ] bdba4bb6344f9593
[RUN_ID]   20244080134-bdba4bb6344f9593
[Python]    3.10.22


In [2]:
import re
import bs4
from bs4 import BeautifulSoup

print("bs4 version:", bs4.__version__)

bs4 version: 4.15.0


## 实验 1:正则表达式的基础匹配

本实验开始导入 `re`,在指定文本上练习基础元字符、字符类、交替与量词。

**任务 1**:在 `"cat on mat; later: cats and dogs"` 上分别匹配 `\d+`、`\s+` 和 `\bcat\b`。

- `\d+`:一个或多个数字——文本中没有数字,返回空列表;
- `\s+`:一个或多个空白字符——文本中有 6 处空格;
- `\bcat\b`:`\b` 是单词边界,只匹配独立的单词 `cat`;`cats` 中 `cat` 与 `s` 同为单词字符、其间不存在边界,因此不匹配。

In [3]:
s = "cat on mat; later: cats and dogs"
print("文本:", repr(s))
print(r"\d+    ->", re.findall(r"\d+", s))
print(r"\s+    ->", re.findall(r"\s+", s))
print(r"\bcat\b ->", re.findall(r"\bcat\b", s))

文本: 'cat on mat; later: cats and dogs'
\d+    -> []
\s+    -> [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b -> ['cat']


**任务 2**:在 `"cat dog bird CAT"` 上使用 `cat|dog`;对比 `re.findall(r"a|aa|aaa", "aaaa")` 与 `re.findall(r"aaa|aa|a", "aaaa")` 的结果。

- `cat|dog` 默认区分大小写,只匹配小写的 `cat` 与 `dog`(`CAT` 不匹配);
- 交替 `|` 从左到右依次尝试各分支:前一种写法在每个位置都先命中单个 `a`,得到 4 个 `a`;后一种写法先命中最长的 `aaa`,剩余部分再匹配一个 `a`。

In [4]:
s = "cat dog bird CAT"
print(r"cat|dog ->", re.findall(r"cat|dog", s))

print(r"a|aa|aaa ->", re.findall(r"a|aa|aaa", "aaaa"))
print(r"aaa|aa|a ->", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog -> ['cat', 'dog']
a|aa|aaa -> ['a', 'a', 'a', 'a']
aaa|aa|a -> ['aaa', 'a']


**任务 3**:在 `"ct cat caat caaat"` 上分别使用 `ca*t`、`ca+t`、`ca?t`、`ca{2}t` 和 `ca{1,2}t`,输出匹配列表并比较各量词的含义。

In [5]:
s = "ct cat caat caaat"
patterns = {
    r"ca*t":     "a 出现 0 次或多次 (*)",
    r"ca+t":     "a 出现 1 次或多次 (+)",
    r"ca?t":     "a 出现 0 次或 1 次 (?)",
    r"ca{2}t":   "a 恰好出现 2 次 ({2})",
    r"ca{1,2}t": "a 出现 1 到 2 次 ({1,2})",
}
for p, desc in patterns.items():
    print(f"{p:12s} {desc:22s} ->", re.findall(p, s))

ca*t         a 出现 0 次或多次 (*)        -> ['ct', 'cat', 'caat', 'caaat']
ca+t         a 出现 1 次或多次 (+)        -> ['cat', 'caat', 'caaat']
ca?t         a 出现 0 次或 1 次 (?)      -> ['ct', 'cat']
ca{2}t       a 恰好出现 2 次 ({2})       -> ['caat']
ca{1,2}t     a 出现 1 到 2 次 ({1,2})   -> ['cat', 'caat']


从输出可见:`*` 允许 0 个 `a`,故 `ct` 也匹配;`+` 要求至少 1 个 `a`,`ct` 被排除;`?` 最多 1 个 `a`,故 `caat`、`caaat` 均不匹配;`{2}` 只匹配恰好 2 个 `a` 的 `caat`;`{1,2}` 匹配 1~2 个 `a`,得到 `cat` 与 `caat`。

## 实验 2:分组、回溯引用与替换

本实验练习捕获组 `(...)`、后向引用 `\1` 以及 `re.sub()` 的替换功能。

**任务 1**:在 `" 联系:alice@example.com;备选:bob@test.org"` 中用捕获组提取每个邮箱地址的用户名和域名。

模式 `(\w+)@([\w.]+)`:`\w+` 捕获 `@` 前的用户名,`[\w.]+` 捕获 `@` 后的域名(允许点号)。

In [6]:
s = " 联系:alice@example.com;备选:bob@test.org"
pairs = re.findall(r"(\w+)@([\w.]+)", s)
print("捕获结果:", pairs)
for user, domain in pairs:
    print(f"用户名 = {user}, 域名 = {domain}")

捕获结果: [('alice', 'example.com'), ('bob', 'test.org')]
用户名 = alice, 域名 = example.com
用户名 = bob, 域名 = test.org


**任务 2**:在 `"cat cat, dog cat, bird bird"` 中用后向引用找出连续重复的单词。

模式 `\b(\w+)\s+\1\b`:`\1` 引用第一个捕获组的内容,要求空白之后的单词与前面完全相同,从而匹配连续重复的单词。

In [7]:
s = "cat cat, dog cat, bird bird"
print("连续重复的单词:", re.findall(r"\b(\w+)\s+\1\b", s))

连续重复的单词: ['cat', 'bird']


**任务 3**:把 `"2024-05-01\n2024/05/02\n2024.05.03"` 中的日期从"年-月-日"统一转换为"月-日-年",保留原有行数。

模式 `(\d{4})[-/.](\d{2})[-/.](\d{2})` 用三个捕获组分别捕获年、月、日,分隔符字符类 `[-/.]` 兼容 `-`、`/`、`.` 三种写法;替换表达式 `\2-\3-\1` 重排为"月-日-年"。`re.sub` 逐处替换,不改变换行。

In [8]:
s = "2024-05-01\n2024/05/02\n2024.05.03"
converted = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", s)
print("转换前:")
print(s)
print()
print("转换后:")
print(converted)
print()
print("行数:", len(s.splitlines()), "->", len(converted.splitlines()))

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后:
05-01-2024
05-02-2024
05-03-2024

行数: 3 -> 3


**任务 4**:把 `" 联系我:123-456-7890 或 987.654.3210"` 中的电话号码统一为 `(123) 456-7890` 形式。

模式 `(\d{3})[-.](\d{3})[-.](\d{4})` 匹配 `-` 或 `.` 分隔的三段数字,替换表达式 `(\1) \2-\3` 生成统一格式。

In [9]:
s = " 联系我:123-456-7890 或 987.654.3210"
print("转换后:", re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", s))

转换后:  联系我:(123) 456-7890 或 (987) 654-3210


## 实验 3:使用 BeautifulSoup 解析 HTML

使用题目给定的 `html_doc` 字符串构建解析树,不依赖网页下载。

**任务 1**:用 `BeautifulSoup(html_doc, "html.parser")` 构建解析树,输出页面标题(`title`)的文本,以及 `h1` 的标签名与文本。

In [10]:
html_doc = """
<html><head><title> 实验课程 </title></head><body>
<h1> 数据分析 </h1>
<p class="course"> 第六讲:正则表达式 </p>
<p class="course"> 第七讲:结构化数据 </p>
<a href="https://example.org/lecture06"> 讲义六 </a>
<a href="https://example.org/lecture07"> 讲义七 </a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("解析树类型:", type(soup).__name__)
print("title 标签名:", soup.title.name)
print("title 文本:", repr(soup.title.get_text()))
print("h1 标签名:", soup.h1.name)
print("h1 文本:", repr(soup.h1.get_text()))
print("h1 文本(strip):", soup.h1.get_text(strip=True))

解析树类型: BeautifulSoup
title 标签名: title
title 文本: ' 实验课程 '
h1 标签名: h1
h1 文本: ' 数据分析 '
h1 文本(strip): 数据分析


**任务 2**:用 `find_all()` 找出所有 `class="course"` 的段落和所有链接;逐项输出链接文字与 `href` 属性。

In [11]:
courses = soup.find_all("p", class_="course")
print("class='course' 的段落数:", len(courses))
for p in courses:
    print("段落文本:", p.get_text())

links = soup.find_all("a")
print("链接数:", len(links))
for a in links:
    print(f"链接文字 = {a.get_text()}, href = {a.get('href')}")

class='course' 的段落数: 2
段落文本:  第六讲:正则表达式 
段落文本:  第七讲:结构化数据 
链接数: 2
链接文字 =  讲义六 , href = https://example.org/lecture06
链接文字 =  讲义七 , href = https://example.org/lecture07


**任务 3**:用 `get_text()` 提取页面可见文本,比较整个文档、单个段落和单个链接的文本结果。

`get_text()` 返回节点下所有文本(不含标签):整个文档把所有节点的文本拼接在一起;单个段落/链接只含自身的文本。相邻节点的文本默认直接相连,可用 `separator` 指定分隔符、`strip=True` 去除首尾空白。

In [12]:
print("整个文档的文本:", repr(soup.get_text()))
print("单个段落的文本:", repr(soup.p.get_text()))
print("单个链接的文本:", repr(soup.a.get_text()))
print("整个文档(separator='|', strip=True):", soup.get_text(separator="|", strip=True))

整个文档的文本: '\n 实验课程 \n 数据分析 \n 第六讲:正则表达式 \n 第七讲:结构化数据 \n 讲义六 \n 讲义七 \n\n'
单个段落的文本: ' 第六讲:正则表达式 '
单个链接的文本: ' 讲义六 '
整个文档(separator='|', strip=True): 实验课程|数据分析|第六讲:正则表达式|第七讲:结构化数据|讲义六|讲义七
